# 03 · Modelo M1: TF-IDF + regresión logística

Línea base clásica del proyecto. Se elige porque los modelos lineales sobre TF-IDF son competitivos en textos cortos, se entrenan en CPU en pocos minutos y sus coeficientes son interpretables.

**Formulación:** plana. Un solo clasificador predice la etiqueta conjunta `Category:Misconception` (64 clases) y sus probabilidades dan directamente el top 3.

**Salidas:**
- `models/m1_tfidf_lr.joblib` (fuera de Git; se comparte por Drive)
- `results/tuning/tuning_M1_tfidf_lr.csv`
- `results/predicciones/pred_M1_tfidf_lr.csv` (split `test`)
- Fila de M1 en `results/tiempos.csv`

> La evaluación detallada sobre el conjunto de prueba es tarea de Persona 2. Aquí solo se usa validación para elegir hiperparámetros.

In [1]:
import sys
import time
import warnings
from pathlib import Path

RAIZ = Path.cwd()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=UserWarning)

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

from src import preprocesamiento as pp
from src.clasificadores import SVMSoftmax
from src.metricas import resumen_metricas, top_k
from src.utilidades import guardar_predicciones, guardar_tuning, registrar_tiempo

NOMBRE = "M1_tfidf_lr"
print("scikit-learn", sklearn.__version__)

scikit-learn 1.7.2


## 1. Datos

Se usa la partición común (`data/splits/particion.csv`) y `preparar_dataframe`, la misma función que usa la aplicación.

In [2]:
df = pp.cargar_train_clean()
datos = pp.aplicar_particion(df)
respuestas, preguntas, misc_por_pregunta = pp.cargar_reglas()

train = pp.preparar_dataframe(datos["train"], respuestas, preguntas)
val = pp.preparar_dataframe(datos["val"], respuestas, preguntas)
test = pp.preparar_dataframe(datos["test"], respuestas, preguntas)

COLUMNAS = ["explicacion_m1", "pregunta_str", "pregunta_respuesta", "correcta_str"]
print({k: len(v) for k, v in datos.items()})
train[COLUMNAS + ["target"]].head(3)

{'train': 25688, 'val': 3668, 'test': 7340}


,explicacion_m1,pregunta_str,pregunta_respuesta,correcta_str,target
0,1 / 3 because 6 over 9 is 2 thirds and 1 third...,31772,31772|1/3,yes,True_Correct:NA
1,1 out of every 3 isn't coloured,31772,31772|1/3,yes,True_Correct:NA
2,1 third because 6 over 9 is 2 thirds and 1 thi...,31772,31772|1/3,yes,True_Correct:NA


## 2. Representación

| Bloque | Columna | Detalle |
|---|---|---|
| TF-IDF de palabras | `explicacion_m1` | n-gramas 1–2, `min_df=2`, `sublinear_tf`. El patrón de tokens conserva números, fracciones (`3/9`) y símbolos, que el patrón por defecto descarta |
| TF-IDF de caracteres | `explicacion_m1` | n-gramas 3–5 (`char_wb`), robustos a errores ortográficos |
| One-hot | `pregunta_str`, `pregunta_respuesta`, `correcta_str` | Pregunta, opción elegida y corrección derivada: el análisis exploratorio mostró que cada idea errónea depende fuertemente de la pregunta |

No se eliminan palabras vacías: términos como *not*, *more* o *than* distinguen razonamientos correctos de erróneos.

In [3]:
PATRON_TOKENS = r"(?u)\d+(?:[./]\d+)*|\w+|[^\w\s]"

def crear_vectorizador(usar_caracteres=True, max_palabras=20000, max_caracteres=30000):
    bloques = [
        ("palabras", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                                     token_pattern=PATRON_TOKENS, max_features=max_palabras), "explicacion_m1"),
    ]
    if usar_caracteres:
        bloques.append(("caracteres", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                                                       sublinear_tf=True, max_features=max_caracteres), "explicacion_m1"))
    bloques.append(("categoricas", OneHotEncoder(handle_unknown="ignore"),
                    ["pregunta_str", "pregunta_respuesta", "correcta_str"]))
    return ColumnTransformer(bloques)

vectorizadores = {"palabras+caracteres": crear_vectorizador(True), "solo palabras": crear_vectorizador(False)}
matrices = {}
for nombre, vec in vectorizadores.items():
    X_tr = vec.fit_transform(train[COLUMNAS])
    matrices[nombre] = (X_tr, vec.transform(val[COLUMNAS]))
    print(f"{nombre}: {X_tr.shape[1]:,} características")

palabras+caracteres: 38,334 características


solo palabras: 20,077 características


## 3. Ajuste de hiperparámetros (validación)

Se evalúa cada combinación en el split de validación con macro-F1 (métrica principal de los objetivos), exactitud y MAP@3 (métrica del reto). Cada modelo se mide **sin** y **con** la máscara de etiquetas, que descarta las etiquetas incoherentes con la corrección de la respuesta o con las ideas erróneas posibles de la pregunta.

In [4]:
clases_train = np.array(sorted(train["target"].unique()))
mascara_val = pp.mascara_etiquetas(clases_train, val["QuestionId"], val["es_correcta"], misc_por_pregunta)

def evaluar(clf, X_val):
    proba = clf.predict_proba(X_val)
    assert list(clf.classes_) == list(clases_train)
    sin = resumen_metricas(val["target"], top_k(proba, clf.classes_)[0])
    con_m = proba * mascara_val
    con_m = con_m / con_m.sum(axis=1, keepdims=True)
    con = resumen_metricas(val["target"], top_k(con_m, clf.classes_)[0])
    return sin, con

resultados = []

def registrar(modelo, representacion, params, clf, X_tr, X_val):
    inicio = time.time()
    clf.fit(X_tr, train["target"])
    duracion = time.time() - inicio
    sin, con = evaluar(clf, X_val)
    for mascara, met in [(False, sin), (True, con)]:
        resultados.append({"modelo": modelo, "representacion": representacion, **params,
                           "mascara": mascara, **{k: round(v, 4) for k, v in met.items()},
                           "segundos_entrenamiento": round(duracion, 1)})
    print(f"{modelo:<18} {representacion:<20} {params} -> F1 {con['f1_macro']:.4f} | MAP@3 {con['map_at_3']:.4f} ({duracion:.0f} s)")

In [5]:
# Regresión logística: regularización C y pesos de clase
X_tr, X_val = matrices["palabras+caracteres"]
for C in [0.5, 1, 2, 4, 8, 16, 32]:
    for peso in [None, "balanced"]:
        clf = LogisticRegression(C=C, class_weight=peso, max_iter=3000)
        registrar("regresion_logistica", "palabras+caracteres", {"C": C, "class_weight": str(peso)}, clf, X_tr, X_val)

regresion_logistica palabras+caracteres  {'C': 0.5, 'class_weight': 'None'} -> F1 0.5187 | MAP@3 0.9095 (52 s)


regresion_logistica palabras+caracteres  {'C': 0.5, 'class_weight': 'balanced'} -> F1 0.4186 | MAP@3 0.7751 (24 s)


regresion_logistica palabras+caracteres  {'C': 1, 'class_weight': 'None'} -> F1 0.5327 | MAP@3 0.9121 (56 s)


regresion_logistica palabras+caracteres  {'C': 1, 'class_weight': 'balanced'} -> F1 0.4548 | MAP@3 0.8179 (26 s)


regresion_logistica palabras+caracteres  {'C': 2, 'class_weight': 'None'} -> F1 0.5772 | MAP@3 0.9153 (74 s)


...

regresion_logistica palabras+caracteres  {'C': 8, 'class_weight': 'balanced'} -> F1 0.5698 | MAP@3 0.8820 (33 s)


regresion_logistica palabras+caracteres  {'C': 16, 'class_weight': 'None'} -> F1 0.6120 | MAP@3 0.9120 (96 s)


regresion_logistica palabras+caracteres  {'C': 16, 'class_weight': 'balanced'} -> F1 0.5929 | MAP@3 0.8956 (45 s)


regresion_logistica palabras+caracteres  {'C': 32, 'class_weight': 'None'} -> F1 0.6119 | MAP@3 0.9107 (129 s)


regresion_logistica palabras+caracteres  {'C': 32, 'class_weight': 'balanced'} -> F1 0.6045 | MAP@3 0.9006 (47 s)


In [6]:
tabla = pd.DataFrame(resultados)
mejor_lr = tabla[(tabla.modelo == "regresion_logistica") & tabla.mascara].sort_values("f1_macro", ascending=False).iloc[0]
C_lr, peso_lr = float(mejor_lr["C"]), (None if mejor_lr["class_weight"] == "None" else mejor_lr["class_weight"])
print(f"Mejor regresión logística: C={C_lr}, class_weight={peso_lr}")

# ¿Aportan los n-gramas de caracteres? Misma configuración, solo palabras
X_tr_p, X_val_p = matrices["solo palabras"]
registrar("regresion_logistica", "solo palabras", {"C": C_lr, "class_weight": str(peso_lr)},
          LogisticRegression(C=C_lr, class_weight=peso_lr, max_iter=3000), X_tr_p, X_val_p)

Mejor regresión logística: C=8.0, class_weight=None


regresion_logistica solo palabras        {'C': 8.0, 'class_weight': 'None'} -> F1 0.5893 | MAP@3 0.9114 (56 s)


In [7]:
# Variante: SVM lineal. CalibratedClassifierCV no admite clases con menos de 3 ejemplos,
# así que las probabilidades se obtienen con un softmax sobre la función de decisión (src/clasificadores.py).
for C in [0.05, 0.1, 0.5, 1, 2]:
    clf = SVMSoftmax(C=C)
    registrar("svm_lineal", "palabras+caracteres", {"C": C, "class_weight": "None"}, clf, X_tr, X_val)

svm_lineal         palabras+caracteres  {'C': 0.05, 'class_weight': 'None'} -> F1 0.4774 | MAP@3 0.9034 (4 s)


svm_lineal         palabras+caracteres  {'C': 0.1, 'class_weight': 'None'} -> F1 0.5451 | MAP@3 0.9091 (4 s)


svm_lineal         palabras+caracteres  {'C': 0.5, 'class_weight': 'None'} -> F1 0.6041 | MAP@3 0.9110 (5 s)


svm_lineal         palabras+caracteres  {'C': 1, 'class_weight': 'None'} -> F1 0.6040 | MAP@3 0.9087 (6 s)


svm_lineal         palabras+caracteres  {'C': 2, 'class_weight': 'None'} -> F1 0.6020 | MAP@3 0.9038 (8 s)


In [8]:
tabla = pd.DataFrame(resultados)
ruta_tuning = guardar_tuning(tabla, NOMBRE)
print("Guardado en", ruta_tuning.relative_to(RAIZ))
tabla.sort_values(["f1_macro", "map_at_3"], ascending=False).head(12)

Guardado en results/tuning/tuning_M1_tfidf_lr.csv


,modelo,representacion,C,class_weight,mascara,f1_macro,accuracy,map_at_3,segundos_entrenamiento
17,regresion_logistica,palabras+caracteres,8.0,None,True,0.6171,0.8370,0.9139,103.4
16,regresion_logistica,palabras+caracteres,8.0,None,False,0.6171,0.8370,0.9138,103.4
21,regresion_logistica,palabras+caracteres,16.0,None,True,0.6120,0.8329,0.9120,95.8
20,regresion_logistica,palabras+caracteres,16.0,None,False,0.6120,0.8329,0.9118,95.8
25,regresion_logistica,palabras+caracteres,32.0,None,True,0.6119,0.8315,0.9107,128.9
24,regresion_logistica,palabras+caracteres,32.0,None,False,0.6119,0.8315,0.9106,128.9
34,svm_lineal,palabras+caracteres,0.5,None,False,0.6054,0.8372,0.9085,5.1
27,regresion_logistica,palabras+caracteres,32.0,balanced,True,0.6045,0.8130,0.9006,46.9
26,regresion_logistica,palabras+caracteres,32.0,balanced,False,0.6045,0.8130,0.9005,46.9
35,svm_lineal,palabras+caracteres,0.5,None,True,0.6041,0.8372,0.9110,5.1


## 4. Modelo final

Se elige la combinación con mayor macro-F1 en validación (con máscara si la mejora o empata) y se entrena el pipeline completo (vectorizador + clasificador) sobre el split de entrenamiento.

In [9]:
mejor = tabla.sort_values(["f1_macro", "map_at_3", "mascara"], ascending=False).iloc[0]
print(mejor.to_string())

# En caso de empate, el orden anterior prefiere la versión con máscara
usar_mascara = bool(mejor["mascara"])

peso = None if mejor["class_weight"] == "None" else mejor["class_weight"]
if mejor["modelo"] == "regresion_logistica":
    clasificador = LogisticRegression(C=float(mejor["C"]), class_weight=peso, max_iter=3000)
else:
    clasificador = SVMSoftmax(C=float(mejor["C"]))

pipeline = Pipeline([
    ("vectorizador", crear_vectorizador(mejor["representacion"] == "palabras+caracteres")),
    ("clasificador", clasificador),
])
inicio = time.time()
pipeline.fit(train[COLUMNAS], train["target"])
t_entrenamiento = time.time() - inicio
print(f"Entrenamiento final: {t_entrenamiento:.1f} s | usar_mascara={usar_mascara}")

modelo                    regresion_logistica
representacion            palabras+caracteres
C                                         8.0
class_weight                             None
mascara                                  True
f1_macro                               0.6171
accuracy                                0.837
map_at_3                               0.9139
segundos_entrenamiento                  103.4


Entrenamiento final: 99.8 s | usar_mascara=True


In [10]:
# Se guardan los coeficientes en float32 para reducir el tamaño del archivo a la mitad
clf_final = pipeline.named_steps["clasificador"]
clf_final = getattr(clf_final, "svm_", clf_final)
if hasattr(clf_final, "coef_"):
    clf_final.coef_ = clf_final.coef_.astype(np.float32)
    clf_final.intercept_ = clf_final.intercept_.astype(np.float32)

artefacto = {
    "pipeline": pipeline,
    "modelo": NOMBRE,
    "usar_mascara": usar_mascara,
    "hiperparametros": {"clasificador": mejor["modelo"], "representacion": mejor["representacion"],
                        "C": float(mejor["C"]), "class_weight": str(peso)},
    "metricas_validacion": {k: float(mejor[k]) for k in ["f1_macro", "accuracy", "map_at_3"]},
    "version_sklearn": sklearn.__version__,
}
pp.DIR_MODELOS.mkdir(exist_ok=True)
ruta_modelo = pp.DIR_MODELOS / "m1_tfidf_lr.joblib"
joblib.dump(artefacto, ruta_modelo, compress=3)
print(f"Modelo guardado en {ruta_modelo.relative_to(RAIZ)} ({ruta_modelo.stat().st_size / 1e6:.1f} MB)")

Modelo guardado en models/m1_tfidf_lr.joblib (8.5 MB)


## 5. Predicciones sobre el conjunto de prueba

Se generan con `src.prediccion.predecir_lote`, la misma función que usa la aplicación, para garantizar que el preprocesamiento sea idéntico. El archivo se entrega a Persona 2 para la evaluación.

In [11]:
from src import prediccion

inicio = time.time()
pred_test = prediccion.predecir_lote(NOMBRE, datos["test"])
t_inferencia = time.time() - inicio

ruta_pred = guardar_predicciones(pred_test, NOMBRE)
registrar_tiempo(NOMBRE, t_entrenamiento, t_inferencia)
print(f"{len(pred_test):,} predicciones en {t_inferencia:.1f} s -> {ruta_pred.relative_to(RAIZ)}")
pred_test.head()

7,340 predicciones en 1.5 s -> results/predicciones/pred_M1_tfidf_lr.csv


,row_id,top1,top2,top3,prob1,prob2,prob3
0,0,True_Neither:NA,True_Correct:NA,True_Misconception:Incomplete,0.972709,0.027099,0.000134
1,16,True_Neither:NA,True_Correct:NA,True_Misconception:WNB,0.669684,0.329845,0.000303
2,19,True_Correct:NA,True_Neither:NA,True_Misconception:Incomplete,0.974348,0.024337,0.001124
3,24,True_Correct:NA,True_Neither:NA,True_Misconception:Incomplete,0.852550,0.146720,0.000542
4,26,True_Correct:NA,True_Neither:NA,True_Misconception:WNB,0.963088,0.036771,0.000075


In [12]:
# Verificación: las predicciones de validación con predecir_lote reproducen la métrica del tuning
pred_val = prediccion.predecir_lote(NOMBRE, datos["val"])
resumen_metricas(datos["val"]["target"], pred_val[["top1", "top2", "top3"]].values)

{'f1_macro': 0.6170523625063089,
 'accuracy': 0.8369683751363141,
 'map_at_3': 0.9138949472918939}

## 6. Interpretación

Términos con mayor peso positivo para algunas ideas erróneas frecuentes (solo aplica si el clasificador final es la regresión logística).

In [13]:
if hasattr(clf_final, "coef_"):
    nombres = pipeline.named_steps["vectorizador"].get_feature_names_out()
    filas = {}
    for etiqueta in ["False_Misconception:Incomplete", "False_Misconception:Additive",
                     "False_Misconception:Duplication", "False_Misconception:Subtraction",
                     "True_Neither:NA", "True_Correct:NA"]:
        coef = clf_final.coef_[list(clf_final.classes_).index(etiqueta)]
        es_texto = np.array([n.startswith("palabras__") for n in nombres])
        top = np.argsort(-np.where(es_texto, coef, -np.inf))[:8]
        filas[etiqueta] = ", ".join(nombres[i].replace("palabras__", "") for i in top)
    display(pd.Series(filas, name="términos con mayor peso").to_frame())

,términos con mayor peso
False_Misconception:Incomplete,"9, not, 24, are not, not shaded, are 9, the 9,..."
False_Misconception:Additive,"5, 4, 10, 9, five, -, 5 =, added"
False_Misconception:Duplication,"numerator and, you multiply, 10 and, 15, 10, b..."
False_Misconception:Subtraction,"away, 1/3 =, because 2/3, 1/3, to subtract, 1/..."
True_Neither:NA,"are shaded, 3 shaded, 3 +, 2 tenths, top to, 5..."
True_Correct:NA,"4, not shaded, by 4, 10/3, 3/5, top number, su..."


## 7. Ejemplo de uso desde la aplicación

In [14]:
prediccion.predecir(NOMBRE, 31772, r"\( \frac{3}{6} \)", "there are 3 white triangles and 6 blue ones so its 3/6")

[{'etiqueta': 'False_Misconception:WNB',
  'category': 'False_Misconception',
  'misconception': 'WNB',
  'prob': 0.953462},
 {'etiqueta': 'False_Neither:NA',
  'category': 'False_Neither',
  'misconception': 'NA',
  'prob': 0.045408},
 {'etiqueta': 'False_Misconception:Incomplete',
  'category': 'False_Misconception',
  'misconception': 'Incomplete',
  'prob': 0.000913}]